In [3]:
#coding:utf-8  
# 手写决策树实现

import numpy as np
import math as mt
from collections import defaultdict

class DecisionTree(object):
    def __init__(self):
        pass

#   计算概率
    def _getDistribution(self,dataArray):
        # dict  map
        distribution = defaultdict(float)
        m, n = np.shape(dataArray)
        for line in dataArray:
            print(line[-1])
            distribution[line[-1]] += 1.0/m
        # 每一个分类号出现的概率  yes 5/14  no 9/14
        return distribution

#   计算信息熵
    def _entropy(self, dataArray):
        ent = 0.0
        distribution=self._getDistribution(dataArray)

        for key, prob in distribution.items():
            ent -= prob * mt.log(prob, 2)
        return ent

    def _conditionEntropy(self, dataArray, colIdx):
        valueCnt = defaultdict(int)
        m, n = np.shape(dataArray)
        # 条件熵
        condEnt = 0.0
        uniqueValues = np.unique(dataArray[:, colIdx])
        for oneValue in uniqueValues:
            oneData = dataArray[dataArray[:, colIdx] == oneValue]
            # 信息熵
            oneEnt = self._entropy(oneData)
            # 第一列值为teenager的概率
            prob = float(np.shape(oneData)[0]) / m
            # 概率*信息熵
            condEnt += prob * oneEnt
        return condEnt

    def _infoGain(self, dataArray, colIdx, baseEnt):
        condEnt = self._conditionEntropy(dataArray, colIdx)
        # 信息增益
        return baseEnt-condEnt

    def _chooseBestProp(self,dataArray):
        m, n = np.shape(dataArray)
        bestProp = -1
        bestInfoGain = 0
        # 计算分类号信息熵
        baseEnt = self._entropy(dataArray)
        # [0-4)
        for i in range(n-1):
            # 计算已知第一列数据的信息熵  条件熵
            infoGain=self._infoGain(dataArray, i, baseEnt)
            if infoGain > bestInfoGain:
                bestProp=i
                bestInfoGain=infoGain
        return bestProp

    def _splitData(self,dataArray,colIdx,splitValue):
        m, n = np.shape(dataArray)

        cols = np.array(range(n)) != colIdx
        rows = (dataArray[:, colIdx] == splitValue)
        print(rows)

        # data=dataArray[rows,:][:,cols]
        #ix_  取rows中指定的行，取cols中指定的列   花式索引
        data = dataArray[np.ix_(rows, cols)]
        return data

    def createTree(self, dataArray):
        # 获取集合形状 (m,n)
        m, n = np.shape(dataArray)
        if len(np.unique(dataArray[:, -1])) == 1:
            return (dataArray[0, -1], 1.0)
        if n == 2:
            distribution = self._getDistribution(dataArray)
            sortProb = sorted(distribution.items(), key=lambda x: x[1], reverse=True)
            return sortProb
        rootNode = {}
        # 选择分类条件的     信息增益  0
        bestPropIdx = self._chooseBestProp(dataArray)

        # 树
        rootNode[bestPropIdx] = {}
        uniqValues = np.unique(dataArray[:, bestPropIdx])
        # 根据第一列的数据来分类
        for oneValue in uniqValues:
            splitDataArray = self._splitData(dataArray, bestPropIdx, oneValue)
            # 要不要把分类出来的这堆数据  进行再次切割   信息熵判断一下
            rootNode[bestPropIdx][oneValue] = self.createTree(splitDataArray)
        return rootNode
    
def loadData():
    # 矩阵
    dataMat = []                 
    fr = open("../../../data/decisiontree.txt")
#     readlines他会一次性将decisiontree.txt文件全部加载到内存的列表中
    lines = fr.readlines()
    for line in lines:
        curLine = line.strip().split('\t')
        dataMat.append(curLine)
    return dataMat


if __name__ == '__main__':
    data = loadData()
    dataarray = np.array(data)
    dt = DecisionTree()
    tree = dt.createTree(dataarray)
    print(tree)

no
no
yes
yes
yes
no
yes
no
yes
yes
yes
yes
yes
no
yes
yes
yes
yes
yes
yes
no
yes
no
no
no
no
yes
yes
no
no
yes
yes
yes
no
yes
yes
yes
no
yes
yes
yes
no
no
no
yes
yes
no
yes
no
yes
no
yes
yes
yes
yes
yes
no
yes
yes
yes
no
yes
yes
yes
no
no
yes
yes
yes
no
[False False  True False False False  True False False False False  True
  True False]
[False False False  True  True  True False False False  True False False
 False  True]
yes
yes
no
yes
no
yes
no
yes
yes
no
yes
no
yes
no
yes
yes
yes
yes
no
no
[ True  True False  True False]
[False False  True False  True]
[ True  True False False False False False  True  True False  True False
 False False]
no
no
no
yes
yes
no
no
yes
no
yes
no
no
no
yes
yes
no
no
yes
no
yes
[ True  True  True False False]
[False False False  True  True]
{0: {'middle': ('yes', 1.0), 'older': {2: {'common': ('yes', 1.0), 'well': ('no', 1.0)}}, 'teenager': {1: {'no': ('no', 1.0), 'yes': ('yes', 1.0)}}}}


In [4]:
#coding:utf-8  

from sklearn import tree as tr

def loadData():
    dataMat = []  
    labels = []               
    fr = open("../../../data/decisiontree.txt")
    lines = fr.readlines()
    for line in lines:
        curLine = line.strip().split('\t')
        dataMat.append(curLine[0:-1])
        labels.append(curLine[len(curLine)-1])
    return dataMat,labels

def string2Float(dataMat,labels):
    string2FloatDict = {"teenager": 0.0,
                        "middle": 1.0,
                        "older": 2.0,
                        "high": 2.0,
                        "medium": 1.0,
                        "low": 0.0,
                        "yes": 1.0,
                        "no": 0.0,
                        "common": 0.0,
                        "well": 1.0
                        }

    def fun1(list):
        return [string2FloatDict.get(elem) for elem in list]

    def fun2(elem):
        return string2FloatDict.get(elem)
    return list(map(fun1, dataMat)), list(map(fun2, labels))


if __name__ == '__main__':
    dataMat, labels = loadData()
    X, Y = string2Float(dataMat, labels)
    print(Y)
    # 衡量纯粹度采用的是信息熵
    clf = tr.DecisionTreeClassifier(criterion="entropy")
    clf = clf.fit(X, Y)
    print(clf)
    #青少年  中等    不是    一般
    test1 = [0, 1, 0, 0]
    test2 = [1, 1, 1, 0]
    # print(clf.predict_proba([test1, test2]))
    # print(clf.predict([test1]))

[0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0]
DecisionTreeClassifier(criterion='entropy')
